In [3]:
#!pip install protobuf==4.23.3

In [9]:
# load libraries

# for data import and manipulation
import json
import numpy as np
import random

# for dataset building
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

# for model building and tracking
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW
from tqdm import tqdm

# for evaluation
from sklearn.metrics import classification_report as sklearn_classification_report

In [5]:
# load the data
with open("/kaggle/input/annotations-social-groups-augmentations/annotations_augmentations.json", "r") as f:
    data = json.load(f)

# subset to only data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# extract augmentations separately
data_synonym_aug = [
    {"sentence": task["augmentations"][0]["sentence"],
     "annotations": task["augmentations"][0]["annotations"]} for task in data_with_annotations]
data_paraphrase_aug = [
    {"sentence": task["augmentations"][-1]["sentence"],
     "annotations": task["augmentations"][-1]["annotations"]} for task in data_with_annotations]

# flatten the data
def flatten_data(data):
    data_flat = []
    for item in data:
        sentence = item["sentence"]
        for ann in item["annotations"]:
            record = {
                "sentence": sentence,
                "group": ann["text"],
                "stance": ann["tag"].lower()[3:]
            }
            data_flat.append(record)
    return data_flat

data_flattened = flatten_data(data_with_annotations)
data_synonym_flattened = flatten_data(data_synonym_aug)
data_paraphrase_flattened = flatten_data(data_paraphrase_aug)

# bind sentences from all methods together
data_bound = [(data_flattened[idx], data_synonym_flattened[idx], data_paraphrase_flattened[idx]) for idx in range(len(data_flattened))]

In [6]:
class EarlyStopping:
    def __init__(self, patience, min_delta=0.0001, save_model=True, path='checkpoint.pt', printoption=False):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_f1 = None
        self.best_epoch = None
        self.early_stop = False
        self.path = path
        self.printoption = printoption
        self.save_model = save_model

    def __call__(self, current_f1, model, epoch):
        if self.best_f1 is None:
            self.best_f1 = current_f1
            self.best_epoch = epoch+1
            self.save_checkpoint(current_f1, model)
        elif current_f1 < self.best_f1 - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            if current_f1 > self.best_f1:
                self.save_checkpoint(current_f1, model)
                self.best_f1 = current_f1
                self.best_epoch = epoch+1
                self.counter = 0

    def save_checkpoint(self, current_f1, model):
        if self.save_model:
            torch.save(model.state_dict(), self.path)
        if self.printoption:
            print(f'Validation F1 increased ({self.best_f1:.6f} --> {current_f1:.6f}).  Saving model ...')

class StanceNLIDataset(Dataset):
    def __init__(self, raw_data, tokenizer, max_len, label2id):
        self.dataset = []

        for item in raw_data:
            sentence = item["sentence"]
            target = item["group"]
            gold_stance = item["stance"]
            
            hypotheses = {
                "pos": f"The text is positive towards {target}.",
                "neg": f"The text is negative towards {target}.",
                "neutral": f"The text is neutral, or contains no stance, towards {target}."
            }

            for stance, hypothesis in hypotheses.items():
                label_text = "entailment" if stance == gold_stance else "not_entailment"

                encoding = tokenizer(
                    sentence,
                    hypothesis,
                    truncation=True,
                    padding="max_length",
                    max_length=max_len,
                    return_tensors="pt"
                    )
                self.dataset.append({
                    "gold_stance": gold_stance,
                    "input_ids": encoding["input_ids"].squeeze(0),
                    "attention_mask": encoding["attention_mask"].squeeze(0),
                    "label": label2id[label_text]
                    })
                

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

def evaluate_nli_stance(model, data):
    model.eval()
    all_preds = []
    all_labels = []

    for item in data:
        sentence = item["sentence"]
        target = item["group"]
        gold_stance = item["stance"]
        
        hypotheses = {
            "pos": f"The text is positive towards {target}.",
            "neg": f"The text is negative towards {target}.",
            "neutral": f"The text is neutral, or contains no stance, towards {target}."
            }

        # Tokenize all 3 hypotheses as a batch
        inputs = tokenizer(
            [sentence]*3,
            list(hypotheses.values()),
            return_tensors="pt",
            padding=True,
            truncation=True
            )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            entail_probs = probs[:, 0].tolist() # 0 is the entailment index

        # choose hypothesis with highest entailment probability
        predicted_stance = list(hypotheses.keys())[entail_probs.index(max(entail_probs))]

        all_labels.append(gold_stance)
        all_preds.append(predicted_stance)

    return all_labels, all_preds

# NLI Model (Laurer, 2024)
## CV of Original Model

In [7]:
# set seeds to ensure reproducibility of all remaining tasks
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [10]:
# create list to store cv results in
num_folds = 5
metrics_original_data = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 10
lr = 4e-05
weight_decay = 0.3

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data_bound]
for fold, (train_idx, val_idx) in enumerate(kf.split(data_bound, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id

    # create a new early stopping object
    checkpoint_path = f"/kaggle/working/no_aug_{fold}.pt"
    early_stopper = EarlyStopping(patience=3, min_delta=0.0001, save_model=True, path=checkpoint_path, printoption=False)
    
    # split the data
    train_fold_data = [data_bound[i] for i in train_idx]
    val_fold_data = [data_bound[i] for i in val_idx]

    # create actual training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
    
        true_labels, pred_labels = evaluate_nli_stance(model, val_data)
        metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)
        val_f1 = metrics["macro avg"]["f1-score"]

        early_stopper(val_f1, model, epoch)
        if early_stopper.early_stop:
            print("Early stopping triggered.")
            break

    # load the best model
    model.load_state_dict(torch.load(f"/kaggle/working/no_aug_{fold}.pt", map_location=device))
    model.to(device)
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_original_data.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)

with open("/kaggle/working/metrics_original_data.json", "w") as f:
    json.dump(metrics_original_data, f)

config.json: 0.00B [00:00, ?B/s]

2025-11-29 12:14:27.435382: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764418467.620695      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764418467.670426      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Epoch 1/10


Training: 100%|██████████| 271/271 [01:30<00:00,  2.99it/s, loss=0.0549]


...

Average training loss: 0.0892
Epoch 3/10


Training: 100%|██████████| 271/271 [01:32<00:00,  2.94it/s, loss=0.00107] 


Average training loss: 0.0376
Epoch 4/10


Training: 100%|██████████| 271/271 [01:31<00:00,  2.96it/s, loss=0.00235] 


Average training loss: 0.0221
Early stopping triggered.


## CV of Oversampling of Negative Class Only

In [26]:
# create list to store cv results in
num_folds = 5
metrics_aug_only_negative = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 10
lr = 4e-05
weight_decay = 0.3

# set the oversample proportion
oversample_prop = 0.5

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data_bound]
for fold, (train_idx, val_idx) in enumerate(kf.split(data_bound, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id

    # create a new early stopping object
    checkpoint_path = f"/kaggle/working/aug_only_negative_{fold}.pt"
    early_stopper = EarlyStopping(patience=3, min_delta=0.0001, save_model=True, path=checkpoint_path, printoption=False)
    
    # split the data
    train_fold_data = [data_bound[i] for i in train_idx]
    val_fold_data = [data_bound[i] for i in val_idx]

    # get the original training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    
    # oversample the minority class
    neg_ann = [r for r in train_fold_data if r[0]["stance"] == "neg"]
    neg_extra = resample(neg_ann, replace=False, n_samples=int(len(neg_ann)*oversample_prop), random_state=0)
    train_data_neg_aug = [task[2] for task in neg_extra]
    train_dataset_neg_aug = StanceNLIDataset(train_data_neg_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_balanced = ConcatDataset([train_dataset, train_dataset_neg_aug])

    # create the data loader
    train_loader = DataLoader(train_dataset_balanced, batch_size=32, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
    
        true_labels, pred_labels = evaluate_nli_stance(model, val_data)
        metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)
        val_f1 = metrics["macro avg"]["f1-score"]

        early_stopper(val_f1, model, epoch)
        if early_stopper.early_stop:
            print("Early stopping triggered.")
            break

    # load the best model
    model.load_state_dict(torch.load(f"/kaggle/working/aug_only_negative_{fold}.pt", map_location=device))
    model.to(device)
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_aug_only_negative.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)

with open("/kaggle/working/metrics_aug_only_negative.json", "w") as f:
    json.dump(metrics_aug_only_negative, f)

Epoch 1/10


Training: 100%|██████████| 285/285 [01:43<00:00,  2.75it/s, loss=0.537] 


Average training loss: 0.2973
Epoch 2/10


Training: 100%|██████████| 285/285 [01:36<00:00,  2.94it/s, loss=0.00442] 


Average training loss: 0.0852
Epoch 3/10


...

Average training loss: 0.0359
Epoch 4/10


Training: 100%|██████████| 285/285 [01:36<00:00,  2.96it/s, loss=0.000236]


Average training loss: 0.0116
Epoch 5/10


Training: 100%|██████████| 285/285 [01:36<00:00,  2.96it/s, loss=0.0633]  


Average training loss: 0.0124
Early stopping triggered.


## CV of Oversampling Proportionally

In [16]:
# create list to store cv results in
num_folds = 5
metrics_aug_prop = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 10
lr = 4e-05
weight_decay = 0.3

# set the oversample proportion
oversample_prop = 0.25

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data_bound]
for fold, (train_idx, val_idx) in enumerate(kf.split(data_bound, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id

    # create a new early stopping object
    checkpoint_path = f"/kaggle/working/aug_prop_{fold}.pt"
    early_stopper = EarlyStopping(patience=3, min_delta=0.0001, save_model=True, path=checkpoint_path, printoption=False)
    
    # split the data
    train_fold_data = [data_bound[i] for i in train_idx]
    val_fold_data = [data_bound[i] for i in val_idx]

    # get the original training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    
    # oversample all classes to a certain proportion and creare datasets
    pos_ann = [r for r in train_fold_data if r[0]["stance"] == "pos"]
    neu_ann = [r for r in train_fold_data if r[0]["stance"] == "neutral"]
    neg_ann = [r for r in train_fold_data if r[0]["stance"] == "neg"]
    pos_extra = resample(pos_ann, replace=False, n_samples=int(len(pos_ann)*oversample_prop), random_state=0)
    neu_extra = resample(neu_ann, replace=False, n_samples=int(len(neu_ann)*oversample_prop), random_state=0)
    neg_extra = resample(neg_ann, replace=False, n_samples=int(len(neg_ann)*oversample_prop), random_state=0)
    train_data_pos_aug = [task[2] for task in pos_extra]
    train_data_neu_aug = [task[2] for task in neu_extra]
    train_data_neg_aug = [task[2] for task in neg_extra]
    train_dataset_pos_aug = StanceNLIDataset(train_data_pos_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_neu_aug = StanceNLIDataset(train_data_neu_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_neg_aug = StanceNLIDataset(train_data_neg_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_balanced = ConcatDataset([train_dataset, train_dataset_pos_aug, train_dataset_neu_aug, train_dataset_neg_aug])

    # create the data loader
    train_loader = DataLoader(train_dataset_balanced, batch_size=32, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
    
        true_labels, pred_labels = evaluate_nli_stance(model, val_data)
        metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)
        val_f1 = metrics["macro avg"]["f1-score"]

        early_stopper(val_f1, model, epoch)
        if early_stopper.early_stop:
            print("Early stopping triggered.")
            break

    # load the best model
    model.load_state_dict(torch.load(f"/kaggle/working/aug_prop_{fold}.pt", map_location=device))
    model.to(device)
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_aug_prop.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)
with open("/kaggle/working/metrics_aug_prop.json", "w") as f:
    json.dump(metrics_aug_prop, f)

Epoch 1/10


Training: 100%|██████████| 338/338 [02:01<00:00,  2.79it/s, loss=0.0401]


Average training loss: 0.2643
Epoch 2/10


Training: 100%|██████████| 338/338 [01:55<00:00,  2.93it/s, loss=0.109]  


Average training loss: 0.0773
Epoch 3/10


...

Average training loss: 0.0279
Epoch 4/10


Training: 100%|██████████| 338/338 [01:54<00:00,  2.95it/s, loss=0.00178] 


Average training loss: 0.0189
Epoch 5/10


Training: 100%|██████████| 338/338 [01:54<00:00,  2.96it/s, loss=0.0013]  


Average training loss: 0.0149
Early stopping triggered.


## CV of Oversampling of All Minority Classes

In [28]:
# create list to store cv results in
num_folds = 5
metrics_aug_negative_neutral = []

# set model name and hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
epochs = 10
lr = 4e-05
weight_decay = 0.3

# create list to store fold metrics in
fold_metrics = {"negative": [], "neutral": [], "positive": [], "macro_average": []}

# create K-Fold splits
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

# loop through the splits
all_labels = [item[0]["stance"] for item in data_bound]
for fold, (train_idx, val_idx) in enumerate(kf.split(data_bound, all_labels)):

    # instantiate a new model instance and create optimizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    label_to_id = model.config.label2id

    # create a new early stopping object
    checkpoint_path = f"/kaggle/working/aug_negative_neutral_{fold}.pt"
    early_stopper = EarlyStopping(patience=3, min_delta=0.0001, save_model=True, path=checkpoint_path, printoption=False)
    
    # split the data
    train_fold_data = [data_bound[i] for i in train_idx]
    val_fold_data = [data_bound[i] for i in val_idx]

    # get the original training data
    train_data_original = [task[0] for task in train_fold_data]
    train_dataset = StanceNLIDataset(train_data_original, tokenizer, max_len=128, label2id=label_to_id)
    
    # oversample all classes to a certain proportion and creare datasets
    neu_ann = [r for r in train_fold_data if r[0]["stance"] == "neutral"]
    neg_ann = [r for r in train_fold_data if r[0]["stance"] == "neg"]
    neu_extra = resample(neu_ann, replace=False, n_samples=int(len(neu_ann)*0.25), random_state=0)
    neg_extra = resample(neg_ann, replace=False, n_samples=int(len(neg_ann)*0.5), random_state=0)
    train_data_neu_aug = [task[2] for task in neu_extra]
    train_data_neg_aug = [task[2] for task in neg_extra]
    train_dataset_neu_aug = StanceNLIDataset(train_data_neu_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_neg_aug = StanceNLIDataset(train_data_neg_aug, tokenizer, max_len=128, label2id=label_to_id)
    train_dataset_balanced = ConcatDataset([train_dataset, train_dataset_neu_aug, train_dataset_neg_aug])

    # create the data loader
    train_loader = DataLoader(train_dataset_balanced, batch_size=32, shuffle=True)

    # unpack validation data (augmentations are not needed)
    val_data = [task[0] for task in val_fold_data]

    # train the model
    model.train()
    
    # loop through epochs
    for epoch in range(epochs):
        
        # print the epoch number
        print(f"Epoch {epoch + 1}/{epochs}")
    
        # initialize training loss for the epoch
        total_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")
    
        # loop through each batch
        for batch in progress_bar:
            
            # move all batch data to respective device
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
    
            # clear the old gradient
            optimizer.zero_grad()
    
            # run data through the model and save the outputs, use mps with mixed precision (16 bit floating point for forward pass)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
                # save the loss and add to the total loss for the epoch
                loss = outputs.loss
                total_loss += loss.item()
    
            # compute gradients by backpropagation, cap gradients to prevent gradient explosion
            loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
    
            # update the model weights based on the gradient and update the progress bar
            optimizer.step()
            progress_bar.set_postfix(loss=loss.item())
    
        # get the average training loss per batch and print
        avg_loss = total_loss / len(train_loader)
        print(f"Average training loss: {avg_loss:.4f}")
    
        true_labels, pred_labels = evaluate_nli_stance(model, val_data)
        metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)
        val_f1 = metrics["macro avg"]["f1-score"]

        early_stopper(val_f1, model, epoch)
        if early_stopper.early_stop:
            print("Early stopping triggered.")
            break

    # load the best model
    model.load_state_dict(torch.load(f"/kaggle/working/aug_negative_neutral_{fold}.pt", map_location=device))
    model.to(device)
                          
    # run through the test set and generate classification report
    true_labels, pred_labels = evaluate_nli_stance(model, val_data)
    metrics = sklearn_classification_report(true_labels, pred_labels, output_dict=True)

    # append metrics to the fold metrics
    fold_metrics["negative"].append(metrics["neg"]["f1-score"])
    fold_metrics["neutral"].append(metrics["neutral"]["f1-score"])
    fold_metrics["positive"].append(metrics["pos"]["f1-score"])
    fold_metrics["macro_average"].append(metrics["macro avg"]["f1-score"])

# take averages across folds
mean_negative = np.mean(fold_metrics["negative"])
mean_neutral = np.mean(fold_metrics["neutral"])
mean_positive = np.mean(fold_metrics["positive"])
mean_macro_avg = np.mean(fold_metrics["macro_average"])

# take standard deviations
sd_negative = np.std(fold_metrics["negative"])
sd_neutral = np.std(fold_metrics["neutral"])
sd_positive = np.std(fold_metrics["positive"])
sd_macro_avg = np.std(fold_metrics["macro_average"])

# calculate confidence intervals
ci_negative = 1.96 * sd_negative / np.sqrt(5)
ci_neutral = 1.96 * sd_neutral / np.sqrt(5)
ci_positive = 1.96 * sd_positive / np.sqrt(5)
ci_macro_avg = 1.96 * sd_macro_avg / np.sqrt(5)


metrics_aug_negative_neutral.append(
    {
        "negative": {"mean": mean_negative, "sd": sd_negative, "lower": mean_negative-ci_negative, "upper": mean_negative+ci_negative},
        "neutral": {"mean": mean_neutral, "sd": sd_neutral, "lower": mean_neutral-ci_neutral, "upper": mean_neutral+ci_neutral},
        "positive": {"mean": mean_positive, "sd": sd_positive, "lower": mean_positive-ci_positive, "upper": mean_positive+ci_positive},
        "macro_avg": {"mean": mean_macro_avg, "sd": sd_macro_avg, "lower": mean_macro_avg-ci_macro_avg, "upper": mean_macro_avg+ci_macro_avg}
}
)
with open("/kaggle/working/metrics_negative_neutral.json", "w") as f:
    json.dump(metrics_aug_negative_neutral, f)

Epoch 1/10


Training: 100%|██████████| 302/302 [01:48<00:00,  2.79it/s, loss=0.15]  


Average training loss: 0.2839
Epoch 2/10


Training: 100%|██████████| 302/302 [01:42<00:00,  2.94it/s, loss=0.0499] 


Average training loss: 0.0849
Epoch 3/10


...

Average training loss: 0.0353
Epoch 4/10


Training: 100%|██████████| 302/302 [01:42<00:00,  2.95it/s, loss=5.78e-5] 


Average training loss: 0.0202
Epoch 5/10


Training: 100%|██████████| 302/302 [01:42<00:00,  2.96it/s, loss=0.000328]


Average training loss: 0.0109
Early stopping triggered.
